In [ ]:
import os
import uuid
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool, tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph

from prompts import QUESTION_GENERATION_PROMPT

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]



class QuestionGeneratorAgent:
    def __init__(
        self,
        tools: list[BaseTool] | None = None,
        model: str = "gpt-5.1-nano",
        system_prompt: str | None = None,
        api_key: str | None = None,
        temperature: float = 0,
        verbose: bool = False,
        subject_content: str | None = None,
        subject_name: str | None = None,
    ) -> None:

        self.tools = tools or []
        self.system_prompt = system_prompt or QUESTION_GENERATION_PROMPT
        self.verbose = verbose
        self.subject_content = subject_content
        self.subject_name = subject_name
        self._tool_map: dict[str, BaseTool] = {t.name: t for t in self.tools}

        self._llm = ChatOpenAI(
            model=model,
            api_key=api_key or os.environ.get("OPENAI_API_KEY"),
            temperature=temperature,
        )
        self._llm_with_tools = self._llm.bind_tools(self.tools)

        self._checkpointer = MemorySaver()
        self._thread_id: str = str(uuid.uuid4())

        self._graph = self._build_graph()
        self._attempts = 0

    def _agent_node(self, state: AgentState) -> AgentState:
        self._log("[agent] Chamando LLM...")

        messages = state["messages"]

        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=self.system_prompt)] + list(messages)

        response = self._llm_with_tools.invoke(messages)
        self._log(f"[agent] Resposta: {response.content[:80]}...")
        return {"messages": [response]}

    def _tool_node(self, state: AgentState) -> AgentState:
        last_message: AIMessage = state["messages"][-1]
        results: list[ToolMessage] = []

        for call in last_message.tool_calls:
            name, args, call_id = call["name"], call["args"], call["id"]
            self._log(f"[tools] Executando '{name}' | args={args}")

            if name in self._tool_map:
                output = self._tool_map[name].invoke(args)
            else:
                output = f"Erro: ferramenta '{name}' não encontrada."

            results.append(
                ToolMessage(
                    content=str(output), 
                    tool_call_id=call_id, 
                    name=name
                    )
                )

        return {"messages": results}

    def _should_continue(self, state: AgentState) -> str:
        """Roteador: 'tools' se há tool_calls pendentes, 'end' caso contrário."""
        last: AIMessage = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else "end"

    def _build_graph(self) -> CompiledStateGraph:
        graph = StateGraph(AgentState)

        graph.add_node("agent", self._agent_node)
        graph.add_node("tools", self._tool_node)

        graph.add_edge(START, "agent")
        graph.add_conditional_edges(
            "agent",
            self._should_continue,
            {"tools": "tools", "end": END},
        )
        graph.add_edge("tools", "agent")

        return graph.compile(checkpointer=self._checkpointer)

    def invoke(self, user_input: str) -> str:
        """Invocação single-turn: cada chamada usa um thread_id isolado."""
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        state = self._graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
        )
        return state["messages"][-1].content

    def chat(self, user_input: str) -> str:
        """Conversa multi-turn: mantém histórico via MemorySaver + thread_id."""
        config = {"configurable": {"thread_id": self._thread_id}}
        state = self._graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
        )
        return state["messages"][-1].content

    def reset(self) -> None:
        """Inicia uma nova conversa gerando um thread_id fresco."""
        self._thread_id = str(uuid.uuid4())
        self._log(f"[agent] Nova sessão iniciada: thread_id={self._thread_id}")

    def add_tool(self, new_tool: BaseTool) -> None:
        self.tools.append(new_tool)
        self._tool_map[new_tool.name] = new_tool
        self._llm_with_tools = self._llm.bind_tools(self.tools)
        self._graph = self._build_graph()
        self._log(f"[agent] Ferramenta '{new_tool.name}' adicionada.")

    def stream(self, user_input: str):
        """Stream single-turn: cada chamada usa um thread_id isolado."""
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
        for step in self._graph.stream(
            {"messages": [HumanMessage(content=user_input)]},
            config=config,
        ):
            yield step

    @property
    def tool_names(self) -> list[str]:
        """Retorna os nomes das ferramentas registradas."""
        return list(self._tool_map.keys())

    def _log(self, message: str) -> None:
        if self.verbose:
            print(message)

    def __repr__(self) -> str:
        return (
            f"ReactAgent(tools={self.tool_names}, "
            f"thread_id={self._thread_id!r})"
        )


if __name__ == "__main__":
    agent = ReactAgent(
        tools=[calculator, web_search, get_weather],
        verbose=True,
    )
    print(repr(agent))

    resposta = agent.invoke("Quanto é (123 * 456) + 789?")
    print(f"\nResposta: {resposta}")

    # ── 2. Conversa multi-turn ─────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("2. CONVERSA MULTI-TURN")
    print("=" * 60)

    r1 = agent.chat("Olá! Qual é o clima em Recife?")
    print(f"Turno 1: {r1}")

    r2 = agent.chat("E em São Paulo?")
    print(f"Turno 2: {r2}")

    r3 = agent.chat("Qual cidade está mais quente?")
    print(f"Turno 3: {r3}")
